# Notebook 4 — Consolidated Visualizations and New Scenario Analyses
## Mechanical Shield Optimisation — Final Synthesis

**Prerequisites:** Notebooks 1, 2, 3 must have been executed first (they produce the model and data artifacts loaded here). Design-level visualizations (synthetic-data EDA, dynamic degradation table, Sobol sensitivity, surrogate robustness/noise analysis, baseline + interactive Pareto plots) now live in **`01_Design_Visuals.ipynb`** — run that notebook too if you want those plots; this notebook only reloads its saved artifacts (`xgb_models`, `pareto_export`) from disk.

---

### What's in this notebook

| Section | Content | Status |
|---|---|---|
| 1 | Closed-loop integration — Bayesian GP update from D̂ | from original |
| 2 | Pareto results (rebuilt from Notebook 1's saved trials) | from original |
| 3 | **Combined impact scenarios matrix — mixed threats in sequence, multi-zone hits** | **NEW** |
| 4 | **Conclusions per application scenario — code + graphs per scenario** | **NEW** |
| 5 | **Multi-scenario Monte Carlo suite — more simulations, more mission profiles** | **NEW** |
| 6 | Results commentary + final conclusion | from original |


## 0. Setup and load artifacts

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────
import os, sys, pickle, json, warnings
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.metrics import r2_score, mean_absolute_error, accuracy_score, confusion_matrix
import xgboost as xgb

from SALib.sample import saltelli
from SALib.analyze import sobol

warnings.filterwarnings('ignore')
np.random.seed(42)

ROOT = Path('.')
DATA_DIR  = ROOT / 'data'
MODEL_DIR = ROOT / 'models'
REPORT_DIR = ROOT / 'reports'
for d in [DATA_DIR, MODEL_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Setup complete')


In [ ]:
# ── Shared definitions (must mirror Notebook 1) ─────────────────────
MATERIALS = ['B4C', 'SiC', 'Dyneema', 'Kevlar', 'NiTi', 'MRF132DG', 'UHMWPE',
             'HHA', 'ARMOX', 'AA7085']
DENSITY = {'B4C': 2520, 'SiC': 3210, 'Dyneema': 970, 'Kevlar': 1440,
           'NiTi': 6450, 'MRF132DG': 3090, 'UHMWPE': 955,
           'HHA': 7850, 'ARMOX': 7850, 'AA7085': 2810}
PRICE   = {'B4C': 115, 'SiC': 55, 'Dyneema': 90, 'Kevlar': 45,
           'NiTi': 175, 'MRF132DG': 600, 'UHMWPE': 28,
           'HHA': 20, 'ARMOX': 25, 'AA7085': 15}
V50_BASE = {'B4C': 850, 'SiC': 780, 'Dyneema': 620, 'Kevlar': 540,
            'NiTi': 480, 'MRF132DG': 510, 'UHMWPE': 560,
            'HHA': 641, 'ARMOX': 582, 'AA7085': 623}
MATERIAL_TYPE = {'B4C': 'passive', 'SiC': 'passive', 'Dyneema': 'passive',
                 'Kevlar': 'passive', 'NiTi': 'active', 'MRF132DG': 'active',
                 'UHMWPE': 'passive',
                 'HHA': 'passive', 'ARMOX': 'passive', 'AA7085': 'passive'}

THREAT_CLASSES = {
    'small_arms':  {'energy_J': (1_000, 5_000),         'velocity_ms': 800,  'prob': 0.50},
    'fpv_drone':   {'energy_J': (30_000, 80_000),       'velocity_ms': 150,  'prob': 0.15},
    'at_rocket':   {'energy_J': (50_000, 200_000),      'velocity_ms': 300,  'prob': 0.25},
    'tank_apfsds': {'energy_J': (5_000_000, 10_000_000),'velocity_ms': 1700, 'prob': 0.10},
}

FEATURE_COLS = (
    [f'thick_{m}' for m in MATERIALS]
    + [f'count_{m}' for m in MATERIALS]
    + ['n_layers', 'areal_density', 'E_k', 'obliquity']
    + [f'threat_{t}' for t in THREAT_CLASSES]
)

def areal_density(layers):
    return sum(DENSITY[m] * t for m, t in layers)

def layer_cost_per_m2(layers):
    return sum(DENSITY[m] * t * PRICE[m] for m, t in layers)

def lambert_jonas_v50(layers, ad_ref=50.0, ad_exponent=0.45):
    total_t = sum(t for _, t in layers)
    if total_t == 0: return 0.0
    v50_avg = sum(V50_BASE[m] * t for m, t in layers) / total_t
    return v50_avg * (areal_density(layers) / ad_ref) ** ad_exponent

def stop_probability(v50, v_strike, sigma=80.0, noise_std=0.04):
    p = 1.0 / (1.0 + np.exp(-(v50 - v_strike) / sigma))
    return float(np.clip(p + np.random.normal(0, noise_std), 0.0, 1.0))

def encode_layers(layers, threat, obliquity=0.0):
    thick = {m: 0.0 for m in MATERIALS}
    count = {m: 0   for m in MATERIALS}
    for m, t in layers:
        thick[m] += t
        count[m] += 1
    ad  = areal_density(layers)
    e_k = float(np.mean(THREAT_CLASSES[threat]['energy_J']))
    row = (
        [thick[m] for m in MATERIALS]
        + [count[m] for m in MATERIALS]
        + [len(layers), ad, e_k, obliquity]
        + [int(threat == t) for t in THREAT_CLASSES]
    )
    return np.array(row, dtype=np.float32).reshape(1, -1)

print('Shared definitions loaded')


In [ ]:
# ── Load artifacts produced by Notebooks 1-3 ────────────────────────
def _safe_load(path, kind='pickle'):
    p = Path(path)
    if not p.exists():
        print(f'  MISSING: {p}')
        return None
    try:
        if kind == 'pickle':
            with open(p, 'rb') as f:
                return pickle.load(f)
        elif kind == 'json':
            with open(p) as f:
                return json.load(f)
        elif kind == 'csv':
            return pd.read_csv(p)
    except Exception as e:
        print(f'  FAILED to load {p}: {e}')
        return None
    return None

xgb_models     = _safe_load(MODEL_DIR / 'xgb_surrogates.pkl')
gp_bundle      = _safe_load(MODEL_DIR / 'gp_pstop.pkl')
pareto_export  = _safe_load(MODEL_DIR / 'pareto_results.pkl')
lj_constants   = _safe_load(MODEL_DIR / 'lj_constants.json', kind='json')
reward_export  = _safe_load(MODEL_DIR / 'rl_reward_histories.pkl')

ablation_df    = _safe_load(DATA_DIR / 'rl_ablation.csv', kind='csv')
vit_synth_df   = _safe_load(DATA_DIR / 'vit_eval_synthetic.csv', kind='csv')
vit_mat_df     = _safe_load(DATA_DIR / 'vit_eval_material_specific.csv', kind='csv')
vit_per_fam_df = _safe_load(DATA_DIR / 'vit_per_family_accuracy.csv', kind='csv')

for name, obj in [('XGBoost', xgb_models),
                  ('GP', gp_bundle), ('Pareto', pareto_export),
                  ('LJ constants', lj_constants), ('RL rewards', reward_export),
                  ('ablation', ablation_df),
                  ('ViT synthetic eval', vit_synth_df),
                  ('ViT material eval', vit_mat_df),
                  ('ViT per-family',   vit_per_fam_df)]:
    print(f'  {name:30s}: {"OK" if obj is not None else "missing"}')


---
## 1. Mathematical integration — closed loop (Monitoring → Control)

```
Monitoring observation D̂
    → p_obs = 1 - D̂/3
    → Bayesian update of GP surrogate
    → Re-optimise Pareto front
    → New recommended configuration x*
    → RL agent re-parametrised with x*
```

In [ ]:
def damage_to_pobs(D_hat: int) -> float:
    #Pretvori stupanj damage u opaženu p_obs.
    return 1.0 - D_hat / 3.0


def bayesian_update_gp(
    gp_bundle: dict,
    x_new: np.ndarray,
    p_obs: float,
) -> dict:
    gp   = gp_bundle["model"]
    scaler = gp_bundle["scaler"]

    X_existing = gp.X_train_
    y_existing = gp.y_train_

    x_scaled = scaler.transform(x_new.reshape(1, -1))
    X_new_all = np.vstack([X_existing, x_scaled])
    y_new_all = np.append(y_existing, p_obs)

    gp_new = GaussianProcessRegressor(
        kernel=gp.kernel_,
        n_restarts_optimizer=0,
        normalize_y=True,
    )
    gp_new.fit(X_new_all, y_new_all)
    return {"model": gp_new, "scaler": scaler}


def closed_loop_step(
    threat: str,
    D_hat: int,
    x_star: np.ndarray,
    gp_bundle: dict,
) -> dict:
    p_obs = damage_to_pobs(D_hat)

    # Bayesian update GP-a
    gp_updated = bayesian_update_gp(gp_bundle, x_star, p_obs)

    # Provjeri smjer pomaka Pareto fronte
    x_scaled = gp_updated["scaler"].transform(x_star.reshape(1, -1))
    mu_new, sigma_new = gp_updated["model"].predict(x_scaled, return_std=True)

    return {
        "threat": threat,
        "D_hat": D_hat,
        "p_obs": p_obs,
        "GP_posterior_mean": float(mu_new[0]),
        "GP_posterior_std": float(sigma_new[0]),
        "gp_updated": gp_updated,
    }


print("Funkcije zatvorene petlje definirane.")

In [ ]:
# ── End-to-end demo: simulated impact sequence ──────────────────
with open(MODEL_DIR / "gp_pstop.pkl", "rb") as f:
    gp_bundle = pickle.load(f)

# Uzmi best trial iz small_arms Pareto fronte (from the saved Optuna export --
# a plain dict of trial values/params, not a live Study object)
best_trial_dict = max(
    pareto_export["small_arms"],
    key=lambda t: t["values"][0]
)
n = best_trial_dict["params"]["n_layers"]
best_layers = [
    (best_trial_dict["params"][f"mat_{i}"], best_trial_dict["params"][f"t_{i}"])
    for i in range(n)
]
x_star = encode_layers(best_layers, "small_arms").flatten()

print(f"Optimalna konfiguracija x*: {best_layers}")
print(f"  p_stop (Pareto) : {best_trial_dict['values'][0]:.3f}")
print(f"  AD (kg/m²)      : {best_trial_dict['values'][1]:.1f}")
print(f"  Cost (EUR/m2)   : {best_trial_dict['values'][2]:.0f}")

# Simuliraj 5 udara s nasumičnim D
print("\n── Simulacija zatvorene petlje (5 udara) ──")
current_gp = gp_bundle
rows_loop = []

for i in range(5):
    D_hat = int(np.random.choice([0, 1, 2, 3], p=[0.4, 0.3, 0.2, 0.1]))
    result = closed_loop_step("small_arms", D_hat, x_star[:len(gp_bundle["scaler"].mean_)], current_gp)
    current_gp = result["gp_updated"]
    rows_loop.append({
        "Udar": i+1,
        "D̂": D_hat,
        "p_obs": round(result["p_obs"], 3),
        "GP μ (novo)": round(result["GP_posterior_mean"], 4),
        "GP σ (novo)": round(result["GP_posterior_std"], 4),
    })
    print(f"  Udar {i+1}: D̂={D_hat}  p_obs={result['p_obs']:.3f}  "
          f"GP_μ={result['GP_posterior_mean']:.4f}  GP_σ={result['GP_posterior_std']:.4f}")

display(pd.DataFrame(rows_loop))


---
## Pareto results (rebuilt from Notebook 1)

Full baseline + interactive Pareto visualizations now live in
`01_Design_Visuals.ipynb`. Here we only rebuild the tabular `pareto_df`
needed to drive the combined-impact and scenario analyses below.


In [ ]:
# ── Rebuild the Pareto DataFrame from Notebook 1's saved Optuna trials ──
def _format_layers(params):
    '''Return a human-readable layer string from Optuna trial params.'''
    n = params.get('n_layers', 1)
    parts = []
    for i in range(n):
        m = params.get(f'mat_{i}')
        t = params.get(f't_{i}')
        if m is None or t is None:
            continue
        parts.append(f'{m} {t*1000:.1f}mm')
    return ' + '.join(parts)

def _uses_active(params):
    n = params.get('n_layers', 1)
    for i in range(n):
        if MATERIAL_TYPE.get(params.get(f'mat_{i}'), 'passive') == 'active':
            return True
    return False

pareto_rows = []
for threat, trials in pareto_export.items():
    for t in trials:
        p_stop, ad, cost = t['values']
        pareto_rows.append({
            'threat':       threat,
            'p_stop':       p_stop,
            'areal_density':ad,
            'cost_per_m2':  cost,
            'config':       _format_layers(t['params']),
            'n_layers':     t['params'].get('n_layers', 1),
            'active':       _uses_active(t['params']),
        })
pareto_df = pd.DataFrame(pareto_rows)
threats_ordered = list(THREAT_CLASSES.keys())
print(f'Total non-dominated solutions across all threats: {len(pareto_df)}')
display(pareto_df.groupby('threat').size().rename('count'))


---
## 6. NEW — Combined impact scenarios matrix (Priority 2)

Real engagements rarely consist of a single threat hitting a single zone in isolation. This section evaluates two compound scenarios that the original notebook did not address:

### 6A. Mixed threats in sequence
A configuration is hit by a sequence of different threats (e.g. `small_arms → fpv_drone → at_rocket`). Each prior impact degrades the shield, lowering its effectiveness against the next threat. We sweep the 4×4 ordering matrix and report the cumulative `p_stop` through the sequence.

### 6B. Simultaneous hits on different zones
Multiple rounds strike different zones of the shield in the same time window. With **N** independent zones each containing one of the top configurations, what's the probability that **all** zones survive a simultaneous K-fold impact? We treat zones as independent draws from their per-configuration `p_stop` and report the joint survival probability.

In [ ]:
# ── 6A. Mixed threats in sequence — cumulative p_stop matrix ─────────
# Use the best-p_stop Pareto config per threat as the candidate to stress-test.
best_config_per_threat = {}
for threat in threats_ordered:
    sub = pareto_df[pareto_df['threat'] == threat].sort_values('p_stop', ascending=False)
    if len(sub):
        best_config_per_threat[threat] = sub.iloc[0].to_dict()

DEGRADATION_FACTOR = {0: 1.00, 1: 0.85, 2: 0.55, 3: 0.15}

def reconstruct_layers(config_str):
    '''Parse "B4C 10.0mm + SiC 14.5mm" → [("B4C", 0.010), ("SiC", 0.0145)]'''
    out = []
    for part in str(config_str).split(' + '):
        part = part.strip()
        if not part: continue
        try:
            mat, t_str = part.rsplit(' ', 1)
            t_m = float(t_str.replace('mm', '')) / 1000.0
            out.append((mat, t_m))
        except Exception:
            continue
    return out

def simulate_sequence(config_layers, threat_sequence, seed=0):
    '''Apply threats in order; return p_stop history.'''
    rng = np.random.default_rng(seed)
    damage = 0
    history = []
    for k, threat in enumerate(threat_sequence):
        x = encode_layers(config_layers, threat)
        p_base = float(xgb_models['p_stop'].predict(x)[0])
        p_eff = p_base * DEGRADATION_FACTOR[damage]
        breach_prob = max(0.0, 1.0 - p_eff)
        breach = rng.random() < breach_prob
        if breach:
            damage = min(3, damage + 1)
        history.append({'step': k+1, 'threat': threat, 'p_stop_base': p_base,
                        'p_stop_effective': p_eff, 'damage_after': damage,
                        'breach': bool(breach)})
    return history

# Build the 4×4 matrix: rows = which config was deployed, cols = ordering pair
all_pairs = [(t1, t2) for t1 in threats_ordered for t2 in threats_ordered]

sequence_rows = []
for cfg_threat, cfg_info in best_config_per_threat.items():
    layers = reconstruct_layers(cfg_info['config'])
    if not layers:
        continue
    for t1, t2 in all_pairs:
        # Monte Carlo: run 50 sequences and report the average final state
        finals = []
        for seed in range(50):
            hist = simulate_sequence(layers, [t1, t2], seed=seed)
            finals.append(hist[-1]['p_stop_effective'])
        sequence_rows.append({
            'deployed_for':   cfg_threat,
            'config':         cfg_info['config'],
            'threat_seq':     f'{t1} → {t2}',
            't1': t1, 't2': t2,
            'p_stop_after_seq': round(float(np.mean(finals)), 4),
            'std':              round(float(np.std(finals)),  4),
        })
sequence_df = pd.DataFrame(sequence_rows)
print(f'Sequence simulations: {len(sequence_df)} (4 configs × 16 pairs × 50 MC runs each)')

# Heatmap: one panel per deployed-for config; rows = first threat, cols = second threat
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, (cfg_threat, sub) in zip(axes.flat, sequence_df.groupby('deployed_for')):
    pivot = sub.pivot(index='t1', columns='t2', values='p_stop_after_seq').reindex(
        index=threats_ordered, columns=threats_ordered)
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0, vmax=1,
                cbar_kws={'label': 'p_stop after both hits'}, ax=ax)
    ax.set_title(f'Config optimised for: {cfg_threat}\n{sub["config"].iloc[0]}', fontsize=10)
    ax.set_xlabel('Second threat'); ax.set_ylabel('First threat')
plt.suptitle('Combined impact matrix — mixed threats in sequence (4×4 ordering)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORT_DIR / '23_combined_impact_sequence.png', dpi=120, bbox_inches='tight')
plt.show()

sequence_df.to_csv(DATA_DIR / 'combined_impact_sequence.csv', index=False)
print(f'Saved → {DATA_DIR / "combined_impact_sequence.csv"}')


In [ ]:
# ── 6B. Simultaneous hits across multiple zones ─────────────────────
# A shield is divided into N zones. K threats arrive simultaneously, each striking a
# random zone. We compute the probability that no zone is breached, assuming each zone
# is a fresh copy of the config and the per-zone p_stop is independent.

def zone_survival_prob(layers, threat, n_zones=5, k_hits=3, n_sim=2000, seed=0):
    rng = np.random.default_rng(seed)
    x = encode_layers(layers, threat)
    p_stop = float(xgb_models['p_stop'].predict(x)[0])
    p_stop = float(np.clip(p_stop, 0.0, 1.0))
    survived_all_runs = 0
    breach_distribution = []
    for _ in range(n_sim):
        zone_hits = rng.integers(0, n_zones, size=k_hits)
        hits_per_zone = np.bincount(zone_hits, minlength=n_zones)
        # For each zone: survives if all its hits are stopped → p_stop^(# hits)
        breach = False
        zone_breach_count = 0
        for h in hits_per_zone:
            if h == 0: continue
            survives_zone = rng.random() < p_stop ** h
            if not survives_zone:
                breach = True
                zone_breach_count += 1
        breach_distribution.append(zone_breach_count)
        if not breach:
            survived_all_runs += 1
    return {
        'p_stop_per_zone': p_stop,
        'p_shield_survives': survived_all_runs / n_sim,
        'expected_zones_breached': float(np.mean(breach_distribution)),
    }

rows_zone = []
for cfg_threat, cfg_info in best_config_per_threat.items():
    layers = reconstruct_layers(cfg_info['config'])
    if not layers: continue
    for k_hits in [1, 2, 3, 5, 8]:
        for n_zones in [3, 5, 8]:
            result = zone_survival_prob(layers, cfg_threat, n_zones=n_zones, k_hits=k_hits, n_sim=1000, seed=42)
            rows_zone.append({
                'config_for':           cfg_threat,
                'config':               cfg_info['config'],
                'n_zones':              n_zones,
                'k_hits':               k_hits,
                'p_stop_per_zone':      round(result['p_stop_per_zone'], 3),
                'p_shield_survives':    round(result['p_shield_survives'], 3),
                'expected_breaches':    round(result['expected_zones_breached'], 3),
            })
zone_df = pd.DataFrame(rows_zone)

# Surface plot per config: x = k_hits, y = n_zones, z = p_shield_survives
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for ax, cfg_threat in zip(axes.flat, threats_ordered):
    sub = zone_df[zone_df['config_for'] == cfg_threat]
    if sub.empty:
        ax.set_visible(False); continue
    pivot = sub.pivot(index='n_zones', columns='k_hits', values='p_shield_survives')
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn', vmin=0, vmax=1,
                cbar_kws={'label': 'P(shield survives)'}, ax=ax)
    ax.set_title(f'Config for {cfg_threat}\n{sub["config"].iloc[0]}', fontsize=10)
    ax.set_xlabel('Simultaneous hits (k)'); ax.set_ylabel('Zones (N)')
plt.suptitle('Combined impact matrix — simultaneous multi-zone hits', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORT_DIR / '24_combined_impact_zones.png', dpi=120, bbox_inches='tight')
plt.show()

zone_df.to_csv(DATA_DIR / 'combined_impact_zones.csv', index=False)
print(f'\nSaved → {DATA_DIR / "combined_impact_zones.csv"}')
print(f'\nFull zone survival table:')
display(zone_df)


---
## 7. NEW — Conclusions per application scenario (Priority 2)

The original notebook discussed scenarios only in markdown prose (the long commentary at the end). This section turns that prose into **concrete code + graphs per scenario**, with each scenario defined as a *mission profile* — i.e. a distribution over threats and a constraint on areal density (mass budget) and cost.

| Scenario | Threat distribution | AD cap (kg/m²) | Cost cap (€/m²) | Notes |
|----|----|----|----|----|
| **Urban patrol (foot)**           | 90% small_arms, 10% fpv_drone | 40  | 8 000  | Soft armour weight budget |
| **Infantry vehicle (light APC)**  | 60% small_arms, 30% at_rocket, 10% fpv_drone | 120 | 25 000 | Add-on armour |
| **Counter-drone fixed installation** | 80% fpv_drone, 20% small_arms | 60  | 6 000  | Optimised vs FPV |
| **MBT supplementary armour**      | 50% at_rocket, 30% tank_apfsds, 20% small_arms | 250 | 60 000 | Heavy add-on |
| **Field-engineering kit (improv.)** | 70% small_arms, 30% at_rocket | 80 | 10 000 | Cost-sensitive |

For each scenario we compute the **expected p_stop** of every Pareto solution under that scenario's threat mix, filter by the constraints, and rank — yielding a concrete materials recommendation per scenario, with full traceability to the data.

In [ ]:
# ── Define scenarios (mission profiles) ─────────────────────────────
SCENARIOS = {
    'Urban patrol (foot)': {
        'threat_mix': {'small_arms': 0.90, 'fpv_drone': 0.10,
                       'at_rocket':  0.00, 'tank_apfsds': 0.00},
        'AD_max':    40,
        'cost_max':  8_000,
    },
    'Infantry vehicle (light APC)': {
        'threat_mix': {'small_arms': 0.60, 'fpv_drone': 0.10,
                       'at_rocket':  0.30, 'tank_apfsds': 0.00},
        'AD_max':    120,
        'cost_max':  25_000,
    },
    'Counter-drone fixed installation': {
        'threat_mix': {'small_arms': 0.20, 'fpv_drone': 0.80,
                       'at_rocket':  0.00, 'tank_apfsds': 0.00},
        'AD_max':    60,
        'cost_max':  6_000,
    },
    'MBT supplementary armour': {
        'threat_mix': {'small_arms': 0.20, 'fpv_drone': 0.00,
                       'at_rocket':  0.50, 'tank_apfsds': 0.30},
        'AD_max':    250,
        'cost_max':  60_000,
    },
    'Field-engineering kit (improv.)': {
        'threat_mix': {'small_arms': 0.70, 'fpv_drone': 0.00,
                       'at_rocket':  0.30, 'tank_apfsds': 0.00},
        'AD_max':    80,
        'cost_max':  10_000,
    },
}
for name, spec in SCENARIOS.items():
    assert abs(sum(spec['threat_mix'].values()) - 1.0) < 1e-6, name
print(f'{len(SCENARIOS)} scenarios defined.')


In [ ]:
# ── Score every Pareto config against every scenario ────────────────
# A config's *expected* p_stop under a scenario = Σ_t w_t · p_stop(config | t)
# We must re-evaluate each config across all threats (not just its "deployed_for" threat)
# because the surrogate is threat-aware.

configs_evaluated = []
for cfg_idx, row in pareto_df.iterrows():
    layers = reconstruct_layers(row['config'])
    if not layers:
        continue
    p_per_threat = {}
    for t in threats_ordered:
        x = encode_layers(layers, t)
        p_per_threat[t] = float(xgb_models['p_stop'].predict(x)[0])
    # AD and cost are closed-form sums -- computed exactly (the pickle
    # now contains only the p_stop surrogate; see Notebook 1).
    ad_x   = areal_density(layers)
    cost_x = layer_cost_per_m2(layers)
    configs_evaluated.append({
        'config':        row['config'],
        'origin_threat': row['threat'],
        'active':        row['active'],
        'AD':            ad_x,
        'cost':          cost_x,
        **{f'p_{t}': p_per_threat[t] for t in threats_ordered},
    })
configs_eval_df = pd.DataFrame(configs_evaluated).drop_duplicates(subset=['config']).reset_index(drop=True)
print(f'Evaluated {len(configs_eval_df)} unique configs across all 4 threats')
display(configs_eval_df.head(3))


In [ ]:
# ── For each scenario: compute expected p_stop, filter by constraints, rank ──
scenario_results = {}
for scenario_name, spec in SCENARIOS.items():
    df_s = configs_eval_df.copy()
    # Expected p_stop
    df_s['expected_p_stop'] = sum(
        spec['threat_mix'][t] * df_s[f'p_{t}'] for t in threats_ordered
    )
    df_s['feasible'] = (df_s['AD'] <= spec['AD_max']) & (df_s['cost'] <= spec['cost_max'])
    scenario_results[scenario_name] = df_s

fig, axes = plt.subplots(len(SCENARIOS), 1, figsize=(12, 3.2 * len(SCENARIOS)))
if len(SCENARIOS) == 1: axes = [axes]
for ax, (scenario_name, df_s) in zip(axes, scenario_results.items()):
    feasible = df_s[df_s['feasible']].sort_values('expected_p_stop', ascending=False).head(8)
    infeas   = df_s[~df_s['feasible']]
    if not feasible.empty:
        bars = ax.barh(feasible['config'][::-1], feasible['expected_p_stop'][::-1], color='seagreen', alpha=0.85)
        ax.set_xlim(0, 1.05)
        ax.set_xlabel('Expected p_stop under scenario threat mix')
        spec = SCENARIOS[scenario_name]
        ax.set_title(f'{scenario_name}    (AD ≤ {spec["AD_max"]} kg/m²,  cost ≤ €{spec["cost_max"]:,}/m²)', fontsize=10)
        for bar, (_, row) in zip(bars, feasible[::-1].iterrows()):
            ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                    f'AD={row["AD"]:.0f}  €={row["cost"]:,.0f}',
                    va='center', fontsize=8)
    else:
        ax.text(0.5, 0.5, f'No feasible configurations\nfound under constraints.',
                ha='center', va='center', transform=ax.transAxes, fontsize=11)
        ax.set_title(scenario_name, fontsize=10)
    ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(REPORT_DIR / '25_conclusions_per_scenario.png', dpi=120, bbox_inches='tight')
plt.show()

# Concrete recommendation table
print('\n' + '='*100)
print('  RECOMMENDED CONFIGURATION PER SCENARIO')
print('='*100)
recommendations = []
for scenario_name, df_s in scenario_results.items():
    feasible = df_s[df_s['feasible']].sort_values('expected_p_stop', ascending=False)
    if feasible.empty:
        recommendations.append({'scenario': scenario_name, 'recommendation': 'NO FEASIBLE CONFIG',
                                'expected_p_stop': None, 'AD': None, 'cost': None})
    else:
        best = feasible.iloc[0]
        recommendations.append({
            'scenario':        scenario_name,
            'recommendation':  best['config'],
            'expected_p_stop': round(best['expected_p_stop'], 3),
            'AD':              round(best['AD'], 1),
            'cost':            round(best['cost'], 0),
        })
rec_df = pd.DataFrame(recommendations)
display(rec_df)
rec_df.to_csv(DATA_DIR / 'recommendations_per_scenario.csv', index=False)
print(f'Saved → {DATA_DIR / "recommendations_per_scenario.csv"}')


---
## 8. NEW — Multi-scenario Monte Carlo simulation suite

The user requested *more simulations across more scenarios*. Section 7 picks an optimal configuration per scenario under the surrogate's deterministic mean prediction. This section goes further: for each (scenario, recommended config) pair we run a **stochastic mission simulation** — drawing a sequence of impacts from the scenario's threat distribution, applying cumulative damage, and tracking the residual shield integrity over the mission.

The output of this section is a per-scenario survival curve and the distribution of *mission outcomes* across N=500 Monte Carlo runs.

In [ ]:
# ── Multi-scenario Monte Carlo mission simulation ───────────────────
N_HITS_MISSION = 8       # impacts per mission
N_RUNS         = 500     # Monte Carlo runs per scenario

def simulate_mission(layers, threat_mix, n_hits, seed):
    '''One mission: sequence of n_hits draws from threat_mix; cumulative damage.'''
    rng = np.random.default_rng(seed)
    threat_names = list(threat_mix.keys())
    probs        = np.array([threat_mix[t] for t in threat_names])
    probs        = probs / probs.sum()

    integrity = 1.0
    damage = 0
    breaches = 0
    p_eff_trace = []
    for hit in range(n_hits):
        t = rng.choice(threat_names, p=probs)
        x = encode_layers(layers, t)
        p_base = float(xgb_models['p_stop'].predict(x)[0])
        p_eff = p_base * DEGRADATION_FACTOR[damage]
        breach = rng.random() > p_eff
        if breach:
            damage = min(3, damage + 1)
            integrity = max(0.0, integrity - 0.20)
            breaches += 1
        p_eff_trace.append(p_eff)
    return {'final_integrity': integrity, 'final_damage': damage,
            'total_breaches': breaches, 'p_eff_trace': p_eff_trace}

# Run simulations
mission_rows = []
survival_curves = {}
for scenario_name, df_s in scenario_results.items():
    feasible = df_s[df_s['feasible']].sort_values('expected_p_stop', ascending=False)
    if feasible.empty:
        survival_curves[scenario_name] = None
        continue
    best_cfg_str = feasible.iloc[0]['config']
    layers = reconstruct_layers(best_cfg_str)
    if not layers:
        survival_curves[scenario_name] = None
        continue

    runs_p_eff = np.zeros((N_RUNS, N_HITS_MISSION))
    for run in range(N_RUNS):
        result = simulate_mission(layers, SCENARIOS[scenario_name]['threat_mix'],
                                   n_hits=N_HITS_MISSION, seed=run)
        runs_p_eff[run] = result['p_eff_trace']
        mission_rows.append({
            'scenario':        scenario_name,
            'config':          best_cfg_str,
            'run':             run,
            'final_integrity': result['final_integrity'],
            'final_damage':    result['final_damage'],
            'total_breaches':  result['total_breaches'],
        })
    survival_curves[scenario_name] = {
        'config': best_cfg_str,
        'p_eff_mean': runs_p_eff.mean(axis=0),
        'p_eff_p10':  np.percentile(runs_p_eff, 10, axis=0),
        'p_eff_p90':  np.percentile(runs_p_eff, 90, axis=0),
    }

mission_df = pd.DataFrame(mission_rows)
print(f'Total mission simulations run: {len(mission_df)} ({N_RUNS} runs × {len(SCENARIOS)} scenarios)')

# Plot survival curves
fig, ax = plt.subplots(figsize=(11, 5))
colors = plt.cm.tab10.colors
for (scenario_name, curve), color in zip(survival_curves.items(), colors):
    if curve is None:
        continue
    hits = np.arange(1, N_HITS_MISSION + 1)
    ax.plot(hits, curve['p_eff_mean'], marker='o', color=color, label=scenario_name, lw=2)
    ax.fill_between(hits, curve['p_eff_p10'], curve['p_eff_p90'], alpha=0.15, color=color)
ax.set_xlabel('Impact number in mission'); ax.set_ylabel('p_stop (effective)')
ax.set_title(f'Multi-scenario mission survival ({N_RUNS} MC runs / scenario, {N_HITS_MISSION}-hit mission)\n'
             'Solid: mean    Shaded: P10-P90 interval')
ax.legend(loc='lower left', fontsize=9)
ax.set_ylim(0, 1.05); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(REPORT_DIR / '26_multi_scenario_survival.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Mission outcome distributions per scenario ──────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, title, xlabel in zip(
    axes,
    ['final_integrity', 'final_damage', 'total_breaches'],
    ['Final shield integrity', 'Final damage class', 'Total breaches per mission'],
    ['integrity (0..1)', 'damage D ∈ {0,1,2,3}', 'breaches'],
):
    if col == 'final_integrity':
        sns.violinplot(data=mission_df, x='scenario', y=col, ax=ax, inner='quartile',
                       palette='Set2')
    else:
        sns.boxplot(data=mission_df, x='scenario', y=col, ax=ax, palette='Set2')
    ax.set_title(title); ax.set_xlabel(''); ax.set_ylabel(xlabel)
    ax.tick_params(axis='x', rotation=25)
    for lbl in ax.get_xticklabels():
        lbl.set_horizontalalignment('right'); lbl.set_fontsize(8)
plt.suptitle(f'Mission outcome distributions across {N_RUNS} Monte Carlo runs',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORT_DIR / '27_mission_outcome_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

# Aggregate summary table
summary = mission_df.groupby('scenario').agg(
    runs=('run', 'count'),
    mean_integrity=('final_integrity', 'mean'),
    p10_integrity=('final_integrity', lambda s: np.percentile(s, 10)),
    mean_breaches=('total_breaches', 'mean'),
    p90_breaches=('total_breaches', lambda s: np.percentile(s, 90)),
    mission_loss_rate=('final_integrity', lambda s: float((s <= 0.0).mean())),
).round(3)
display(summary)
summary.to_csv(DATA_DIR / 'mission_outcome_summary.csv')
mission_df.to_csv(DATA_DIR / 'mission_simulations.csv', index=False)
print(f'Saved → {DATA_DIR / "mission_outcome_summary.csv"}')
print(f'Saved → {DATA_DIR / "mission_simulations.csv"}')


---
## Results Commentary and Material Recommendations

### Surrogate Models
XGBoost achieves R²=0.9915 for p_stop and R²=0.9993 for mass and cost — excellent across all three targets. The GP achieves R²=0.9899 with uncertainty quantification. The Lambert-Jonas direct validation (R²=0.9977, MAE=0.0167 m/s across 500 noise-free samples) confirms the surrogate faithfully approximates the physical model rather than memorising training noise. Error is uniform across all four threat categories (0.015–0.019 MAE), meaning the model generalises equally well regardless of threat type.

### Feature Importance (Permutation)
The most important finding: **threat_at_rocket (importance=0.517) and areal_density (0.303) dominate**, with thick_B4C third (0.150). This is physically consistent — at at_rocket energy levels (50–200 kJ) total shield mass is more critical than material choice. For small_arms (1–5 kJ) the specific material matters more, but this gets diluted in the global average across all threat classes. The Sobol and permutation results broadly agree on ranking, which validates the surrogate's physical consistency.

> **Note:** XGBoost native gain returns feature names as f17–f20 without labels — this is a known issue when the booster is not passed a `feature_names` list at training time. Permutation importance (sklearn) correctly resolves names from FEATURE_COLS.

---

### Pareto Optimisation Results

**small_arms** — B4C+SiC+NiTi triple dominates the high-protection end (p_stop=1.010). However, B4C 10mm+SiC 15mm+SiC 8mm achieves p_stop=0.977 at only 98.9 kg/m² and **6,874 €/m²** — the best cost-protection trade-off. MRF-132DG only appears in expensive configurations (>47k €/m²).

**FPV drone** — surprising result: **Dyneema 20mm+Dyneema 15mm** achieves p_stop=1.006 at only 33.7 kg/m² and **2,975 €/m²** — by far the lightest and cheapest high-performance configuration across all threats. A single SiC 12.8mm layer reaches p_stop=0.997 for 2,245 €/m². This confirms the FPV energy range (30–80 kJ) does not require heavy ceramics.

**at_rocket** — a single **SiC 24.2mm layer** achieves p_stop=0.998 at 77.4 kg/m² and 4,213 €/m² — an outstanding result. B4C+NiTi delivers similar protection at 4× higher cost. The Dyneema+MRF132DG combination appears but at 46k €/m², which is impractical.

**tank_apfsds** — maximum p_stop=0.056 confirms the physical limitation of lightweight shields against ~8–10 MJ APFSDS projectiles. The cheapest viable option is UHMWPE 8mm+SiC triple for 2,030 €/m², providing minimal but non-zero protection.

---

### Reinforcement Learning
**small_arms agent — avg reward 1.077** — the only agent that learned a genuinely useful policy, outperforming the passive baseline (-0.800) by nearly 2 reward units. The LSTM successfully learned to activate NiTi/MRF-132DG after detecting energy accumulation patterns. **FPV drone, at_rocket, tank_apfsds — all at -0.410** — improved over -0.800 passive but not fully converged. The ablation study confirms RL achieves 0.367 vs -0.800 (no activation) and -0.721 (random), demonstrating clear learning above random policy.

The small_arms agent's superior convergence is explained by the energy range: at 1–5 kJ the agent has meaningful control authority over the outcome, whereas at 5–10 MJ (tank_apfsds) activation makes negligible difference given the shield's fundamental p_stop ceiling of ~0.05.

---

### Multiple Impacts on Same Point
B4C+SiC 18mm: 6 distributed hits vs 4 same-point hits before p_stop<0.5 — same-point impacts are **1.5× more dangerous** for ceramic stacks. SiC+Kevlar: 5 vs 2 — **2.5× more dangerous at the same point**, consistent with brittle ceramic fracture mechanics where a first hit creates micro-cracks that dramatically reduce V50 at the same location. This strongly motivates the MRF-132DG activation strategy: redirecting impact energy away from already-damaged zones.

---

### Active Material Programmable Resistance
**NiTi small_arms**: Δp_stop=+0.475 (passive=0.516 → max_active=0.990) — dramatic improvement, but 2000ms thermal response makes it impractical for single-shot projectile scenarios (bullet transit time ~1ms). Best suited for pre-activation before known threat exposure.

**MRF-132DG small_arms**: Δp_stop=+0.353 with <1ms response — clinically relevant because it can react between consecutive impacts. 10× lower power cost (5 W/cm² vs 50 W/cm²). For FPV drones both materials already achieve p_stop=1.000 in passive state at maximum thickness.

---

### ViT Damage Classification
100% accuracy across all four D classes on the balanced synthetic dataset. This result should be interpreted cautiously — the procedurally generated images are visually distinct by construction (clean noise for D=0, lines for D=1, crack networks for D=2, holes for D=3), making classification trivial. On real damage imagery accuracy would be substantially lower. This is a known limitation documented in Section 5.1.

---

### Sobol Sensitivity Analysis (7 materials including UHMWPE)
**SiC (ST=0.493) and B4C (ST=0.466) are now nearly equal**, compared to the previous 6-material analysis where B4C clearly dominated (ST=0.61). Adding UHMWPE redistributed influence because SiC and UHMWPE share a similar role in the model's feature space (moderate density, intermediate V50). NiTi is third (ST=0.335), confirming the active material's structural importance despite its high cost. Kevlar (ST=0.041) has minimal independent influence.

---

### Robustness — Noise-Augmented Training
**This is the most practically important finding in the entire project.** The original model collapses from R²=0.99 to R²=0.35 at just σ=2% measurement noise. The noise-augmented model (28,000 samples = original + 3 noise levels) maintains:
- R²=0.9915 → **0.9915** at σ=2% (unchanged)
- R²=0.9915 → **0.9895** at σ=5%
- R²=0.9915 → **0.8842** at σ=20%

This near-complete elimination of noise sensitivity is critical for real deployment, where sensor measurements always contain noise. This comparison (see `01_Design_Visuals.ipynb`) is diagnostic only — if noise-robustness is required in production, retrain the single unified surrogate with noise augmentation baked into its own training step, rather than shipping a second parallel model.

---

### Dynamic Degradation Table — Known Issue
p_stop values remain constant across all 4 impacts for small_arms, FPV and at_rocket configurations — only tank_apfsds degrades correctly (0.055 → 0.008). The issue is that `breach` is always False when p_stop_initial > 0.5, so the damage level D never increments. This is a logical bug in the `simulate_degradation()` function: `breach_prob = 1 - p_stop × DEGRADATION_FACTOR[D]` produces near-zero probabilities when p_stop ≈ 1.0. Fix planned: use a minimum breach probability floor or model degradation as deterministic per hit.

---
---

## Material Recommendation by Scenario

Based on the complete Pareto analysis, feature importance, active material tests and cost-performance trade-offs:

### Personal Armor (mass critical, ≤ 10 kg/m²)
**→ Dyneema SB301 or UHMWPE**

Dyneema 20mm+15mm achieves p_stop=1.006 at 33.7 kg/m² and 2,975 €/m² for FPV drones. For small_arms body armor at NIJ IIIA level, a single Dyneema or UHMWPE layer (10–15mm) is the only viable option — ceramics are too heavy. UHMWPE is 3× cheaper than Dyneema with ~10% lower performance.

| Priority | Material | Reason |
|----------|----------|--------|
| 1st | Dyneema SB301 | Best mass-protection ratio |
| 2nd | UHMWPE | Same base polymer, 3× cheaper |
| 3rd | Kevlar KM2 Plus | Heavier but established NIJ standard |

---

### Vehicle Armor (balance of protection, mass, cost)
**→ B4C + SiC dual layer**

B4C 10mm + SiC 15mm + SiC 8mm: p_stop=0.977, 98.9 kg/m², 6,874 €/m² for small_arms. SiC 24.2mm single layer: p_stop=0.998, 77.4 kg/m², 4,213 €/m² for at_rocket. SiC dominates here — it is 2× cheaper than B4C per kg with only marginally lower V50, and the Sobol analysis shows ST scores nearly equal (B4C=0.466, SiC=0.493).

| Priority | Material | Reason |
|----------|----------|--------|
| 1st | SiC (single or dual) | Best cost-performance, cheaper than B4C |
| 2nd | B4C + SiC | Higher absolute protection when cost allows |
| 3rd | B4C + SiC + NiTi | Maximum protection, pre-activated before engagement |

---

### Stationary Protection (cost critical, weight irrelevant)
**→ SiC dominant**

SiC 24.2mm at 4,213 €/m² achieves p_stop=0.998 against at_rocket. For small_arms, SiC 18–24mm provides >0.96 p_stop at 6,000–7,500 €/m². Mass is irrelevant for fixed installations so NiTi can be added for active reinforcement — but at 175 €/kg and 2000ms response, the cost is hard to justify over simply increasing SiC thickness.

| Priority | Material | Reason |
|----------|----------|--------|
| 1st | SiC | Cheapest per protection unit for fixed use |
| 2nd | B4C + SiC | When highest protection is required |
| 3rd | MRF-132DG layer | Add if budget allows, for same-point impact mitigation |

---

### Counter-FPV Drone (fast response, lightweight)
**→ Dyneema or SiC, with MRF-132DG activation layer**

Dyneema 20mm+15mm: p_stop=1.006, 33.7 kg/m², 2,975 €/m² — cheapest viable option. SiC 12.8mm: p_stop=0.997, 40.6 kg/m², 2,245 €/m² — even cheaper. The key addition here is **MRF-132DG as an active layer** — its <1ms response is the only active material that can react within the threat timeline. Δp_stop=+0.353 at 5 W/cm² power cost makes it cost-effective for high-priority assets.

| Priority | Material | Reason |
|----------|----------|--------|
| 1st | Dyneema + MRF-132DG | Lightest high-performance combo |
| 2nd | SiC + MRF-132DG | Slightly heavier but cheaper base |
| 3rd | B4C + SiC | If no active layer is available |

---

### Summary Comparison Table

| Scenario | Best material | p_stop | Mass (kg/m²) | Cost (€/m²) |
|----------|--------------|--------|--------------|-------------|
| Personal armor | Dyneema 20mm+15mm | 1.006 | 33.7 | 2,975 |
| Vehicle — small_arms | B4C 10mm + SiC 15mm + SiC 8mm | 0.977 | 98.9 | 6,874 |
| Vehicle — at_rocket | SiC 24.2mm | 0.998 | 77.4 | 4,213 |
| Stationary — at_rocket | SiC 24.2mm | 0.998 | 77.4 | 4,213 |
| Counter-FPV (active) | Dyneema 35mm + MRF-132DG | ~1.02 | ~45 | ~8,000 |
| APFSDS (max possible) | B4C 15mm + SiC 10mm | 0.056 | 70.5 | 6,193 |

**Overall winner for versatility: SiC** — second-best protection across all scenarios, lowest cost per protection unit, works as both single and dual layer, and its Sobol ST score (0.493) now slightly exceeds B4C (0.466). B4C remains the absolute maximum-protection option when cost is not a constraint.

In [ ]:
# ── Generate requirements.txt ───────────────────────────────────
req_content = """# requirements.txt — ML optimization shield
# Python 3.11+

numpy>=1.26
pandas>=2.2
matplotlib>=3.8
seaborn>=0.13
plotly>=5.20

scikit-learn==1.5.*
xgboost>=2.0
optuna>=3.6

torch>=2.3
torchvision>=0.18

stable-baselines3>=2.3
sb3-contrib
gymnasium>=0.29

transformers>=4.42
datasets>=2.20
accelerate

SALib>=1.5
tensorboard
tqdm
joblib
"""

with open(ROOT / "requirements.txt", "w") as f:
    f.write(req_content)

print("requirements.txt generated.")
print("\nAll artefacts:")
for p in sorted((ROOT / "reports").glob("*.png")):
    print(f"  reports/{p.name}")
for p in sorted((ROOT / "models").glob("*")):
    print(f"  models/{p.name}")
print(f"  data/synthetic_shield.csv")

---
## Conclusion

The developed three-level ML system successfully integrates surrogate modelling, reinforcement
learning and transfer learning damage classification into a coherent closed loop for
optimisation of mechanical shield geometry.

**Key results:**
- XGBoost surrogates achieve R² = 0.99 for p_stop, areal density and cost_per_m², enabling
  ~1000x faster configuration evaluation than direct physical simulations
- Pareto optimisation identified optimal configurations per threat — B4C+SiC
  combinations dominate for small_arms and at_rocket, while tank_apfsds remains physically
  constrained (max p_stop ≈ 0.10) for lightweight shields
- ViT damage classifier, after balancing the dataset across 4 visually distinct classes,
  achieves [UPDATE after new training run]% accuracy
- Sobol analysis confirms the dominant influence of B4C thickness (ST = 0.61) with strong
  interaction effects between layers

**Limitations and directions for future work:**
- Surrogate robustness to noise remains a critical challenge (R² drops to 0.13 at σ=2%) —
  noise-augmented training strategy is required for deployment in real conditions
- Integration of real ballistic data (NIJ Standard-0101.06) is recommended
  for validation of the synthetic model before eventual application

The system demonstrates that combining classical physical models (Lambert-Jonas, Recht-Ipson)
with modern ML techniques enables fast, scalable optimisation of multi-material protective
systems — an approach that transcends the limitations of purely analytical or experimental methods.